# Checkpoint 47: Current-State Dashboard Validation

This notebook reviews the generated dashboard population, frozen human-review plan, timeline metadata, and validation checks. Run `scripts/run_checkpoint47.ps1` first.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DASHBOARD = PROCESSED / 'dashboard'

risk = pd.read_csv(DASHBOARD / 'retention_risk_employees.csv')
policy = pd.read_csv(DASHBOARD / 'current_policy_summary.csv')
metadata = pd.read_csv(DASHBOARD / 'dashboard_metadata.csv')
validation = pd.read_csv(PROCESSED / 'dashboard_current_state_validation.csv')


In [2]:
display(metadata.T)
display(policy)
display(validation)

,0
as_of_date,2026-06-30
workforce_status_label,Current active synthetic workforce
probability_label,Estimated 12-month attrition probability
score_horizon_months,12
model_status_label,Out-of-time tested planning projection
final_test_snapshot,2025-06-30
selected_policy,Budget-constrained expected value
synthetic_data_notice,All people and outcomes in this project are sy...
current_score_caveat,Current scores are planning projections from a...
legacy_score_caveat,Legacy Version 1 full-snapshot scores were in-...


,snapshot_date,eligible_employees,selected_for_human_review,intervention_budget_usd,projected_intervention_spend_usd,projected_expected_prevented_departures,projected_expected_avoided_cost_usd,projected_expected_net_value_usd,human_review_required,automatic_employment_action_permitted
0,2026-06-30,7305,700,1750000.0,1750000.0,35.034285,4.146216e+06,2.396216e+06,True,False


,check,status,observed,requirement,details
0,Dashboard as-of date is explicit and exact,PASS,['2026-06-30'],['2026-06-30'],Every employee row and the UI metadata use one...
1,Dashboard contains current active employees only,PASS,0,0,Terminated employees cannot appear as current ...
2,Active model-eligible population reconciles,PASS,7305,7305,Current scores must cover all and only eligibl...
3,Protected hierarchy employees are excluded,PASS,104,> 0 and absent from risk rows,Department heads and senior managers remain in...
4,Employee dashboard keys are unique,PASS,0,0,Each current employee receives one dashboard row.
5,Frozen review capacity is preserved,PASS,700,700,Checkpoint 47 displays but does not retune the...
6,Current probabilities are finite and bounded,PASS,min=0.017415; max=0.425890,"All finite and inside [0, 1]",The approved calibrated probability scale is p...
7,No future outcome columns are displayed,PASS,[],[],Current outcomes after the as-of date are unkn...
8,Direct personal names are omitted,PASS,[],[],The synthetic review table remains anonymized ...
9,Out-of-time model evidence is retained,PASS,['2025 final test'],['2025 final test'],Performance metrics are not calculated from cu...


In [3]:
population_check = pd.DataFrame({
    'metric': [
        'Dashboard rows',
        'Unique employees',
        'Inactive employees displayed',
        'Selected for human review',
        'Automatic actions permitted',
    ],
    'value': [
        len(risk),
        risk['employee_id'].nunique(),
        risk['employment_status'].ne('Active').sum(),
        risk['selected_for_human_review'].sum(),
        risk['automatic_employment_action_permitted'].sum(),
    ],
})
display(population_check)

,metric,value
0,Dashboard rows,7305
1,Unique employees,7305
2,Inactive employees displayed,0
3,Selected for human review,700
4,Automatic actions permitted,0


In [4]:
department_plan = (
    risk.groupby('department_name', as_index=False)
    .agg(
        eligible_employees=('employee_id', 'count'),
        selected_for_human_review=('selected_for_human_review', 'sum'),
        average_probability=('attrition_probability', 'mean'),
    )
)
display(department_plan.sort_values('selected_for_human_review', ascending=False))

,department_name,eligible_employees,selected_for_human_review,average_probability
1,Engineering,1512,251,0.109443
5,Manufacturing,1808,162,0.127020
4,Information Technology,734,152,0.127715
2,Finance,570,55,0.141591
7,Supply Chain,904,31,0.108829
3,Human Resources,503,23,0.132382
6,Sales,739,18,0.114442
0,Customer Support,535,8,0.158424
